In [2]:
import torch
t = torch.load('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_feature/0.pt')

t.shape

/tmp/ipykernel_253509/1947809781.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  t = torch.load('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_feature/0.pt')


torch.Size([1, 3, 256])

In [1]:
from sam2.build_sam import build_sam2_video_predictor

In [2]:

sam2_checkpoint = "/mnt/petrelfs/share_data/yanziang/sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device='cuda:0')
# for _, param in predictor.named_parameters():
#     # param.requires_grad = False
#     param = param.to(dtype = self.lm.dtype)
sam = predictor
# predictor = predictor.to(self.language_model.device)
# freeze_module(self.sam)

In [3]:
from petrel_client.client import Client
c = Client()
def read_frames_decord_all(
        video_path, client=None, ret_all_frames=False
    ):
    # print("Reading decord")
    if 's3://' in video_path:
        video_bytes = client.get(video_path)
        video_reader = VideoReader(io.BytesIO(video_bytes), num_threads=1)
    else:
        video_reader = VideoReader(video_path, num_threads=1)

    vlen = len(video_reader)
    # print(video_reader[0].shape)
    fps = video_reader.get_avg_fps()
    duration = vlen / float(fps)

    # frames = video_reader.get_batch(frame_indices)  # (T, H, W, C), torch.uint8
    # frames = frames.permute(0, 3, 1, 2)  # (T, C, H, W), torch.uint8
    frame_indices = [i for i in range(0, vlen, 1)]
    all_frames = video_reader.get_batch(frame_indices)  # (T, H, W, C), torch.uint8
    # import pdb;pdb.set_trace()
    all_frames = torch.tensor(all_frames.asnumpy().tolist())
    all_frames = all_frames.permute(0, 3, 1, 2)  # (T, C, H, W), torch.uint8
    if ret_all_frames:
        return all_frames, frame_indices, float(fps), vlen
    else:
        return all_frames[0], frame_indices, float(fps), vlen

def load_and_transform_media_data_video_sam2(data_path,  client, image = False, frame_num = 1 ):
    try:
        if image:
            all_frames = load_image_from_path(data_path, client)
        else:
            all_frames, frame_indices, fps, vlen = read_frames_decord_all(
                data_path, client, ret_all_frames=True
            )
        from torchvision import transforms
        mean = (0.485, 0.456, 0.406)
        std = (0.229, 0.224, 0.225)
        type_transform = transforms.Lambda(lambda x: x.float().div(255.0))
        normalize = transforms.Normalize(mean, std)
        image_size = 1024
        sam2_transform = transforms.Compose(
            [
                type_transform,
                transforms.Resize((image_size, image_size)),
                normalize,
            ]
        )
        if not image:
            # all_frames = all_frames[:16]
            if frame_num > 0:
                all_frames = all_frames[:frame_num]
            else:
                all_frames = all_frames
        all_frames = sam2_transform(all_frames)
        return all_frames
    except Exception as e:
        # logger.info(f"Sam read: Something wrong when read {data_path}")
        raise e

In [3]:
import json
from tqdm import tqdm
import numpy as np
import jsonlines
from decord import VideoReader, cpu
import io
import torch
data = []
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    # with open(json_file_path, 'r', encoding='utf-8') as json_file:
    #     data = json.load(json_file)
    with jsonlines.open(json_file_path, 'r') as f:
        for dd in f:
            data.append(dd)
        print(data[0])
        fin_data = []
        for dd in tqdm(data):
            tt = dd
            fin_data.append(tt)
            # tt['gt_embs'] = 
            video_path = 'pnorm2:s3://' + dd['video']
            if 's3://' in video_path:
                video_bytes = c.get(video_path)
                video_reader = VideoReader(io.BytesIO(video_bytes), num_threads=1)
            else:
                video_reader = VideoReader(video_path, num_threads=1)
        
            vlen = len(video_reader)
            orig_height, orig_width = video_reader[0].shape[0], video_reader[0].shape[1]
            print(orig_height, orig_width)
            sam_gt_bbox = dd['bbox']
            sam_gt_bbox = [ sam_gt_bbox[1] / orig_height, sam_gt_bbox[0] / orig_width, sam_gt_bbox[3] / orig_height, sam_gt_bbox[2] / orig_width ]
            sam_images = load_and_transform_media_data_video_sam2('pnorm2:s3://' + dd['video'], c, False, 1)
            print(sam_gt_bbox)
            inference_state = sam.init_state_images(sam_images, orig_height, orig_width)
            print(orig_height, orig_width)
            sam_gt_embs = sam.get_prompt_embeding(inference_state, None, None, False, sam_gt_bbox[idx], device = self.language_model.device) 
            tt['sam_gt_embs'] = sam_gt_embs
            print(tt)
            break
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_50k.jsonl', '/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_embs.jsonl')

{'video': 'mevis/videos/6084240e75fa.mp4', 'seg_path': 'pnorm2:s3://mevis/masks/289.json', 'bbox': [1003.0, 549.0, 1447.0, 950.0], 'conversations': [{'from': 'human', 'value': 'Visualize the tracking trajectory of panda rolling around in the video.'}, {'from': 'gpt', 'value': 'Answer: <track_begin><tracking> .'}]}


  0%|          | 0/14259 [00:00<?, ?it/s]


NameError: name 'c' is not defined

In [17]:

    
import json
from tqdm import tqdm
import numpy as np
import os
import jsonlines
import torch
data = []
no_feature = []
have_feature = [] 
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    # with open(json_file_path, 'r', encoding='utf-8') as json_file:
    #     data = json.load(json_file)
    with jsonlines.open(json_file_path, 'r') as f:
        for dd in f:
            data.append(dd)
        print(data[0])
        fin_data = []
        for i, dd in tqdm(enumerate(data)):
            tt = dd
            tt['sam_gt_embs'] = f'/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_feature/{i}.pt'
            fin_data.append(dd)
            if os.path.exists(tt['sam_gt_embs']):
                try:
                    feature = torch.load(tt['sam_gt_embs'])
                    # print(feature.shape)
                    if feature.shape != torch.Size([1, 3, 256]):
                        print(dd)
                        raise UnboundLocalError
                    have_feature.append(tt)
                        
                except Exception as e:
                    # print(e)
                    dd['name'] = tt['sam_gt_embs']
                    no_feature.append(dd)
                
            else:
                dd['name'] = tt['sam_gt_embs']
                no_feature.append(dd)
            # break
    print(len(have_feature))
    with open(out_path, 'w', encoding='utf-8') as jsonl_file:
        if isinstance(fin_data, list):
            for item in have_feature:
                jsonl_file.write(json.dumps(item) + '\n')
        else:
            jsonl_file.write(json.dumps(have_feature) + '\n')
    print(len(no_feature))
    with open('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_feature/no_feature.json', 'w') as f:
        json.dump(no_feature, f)
# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_50k.jsonl', f'/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_embs_feature.jsonl')

{'video': 'mevis/videos/6084240e75fa.mp4', 'seg_path': 'pnorm2:s3://mevis/masks/289.json', 'bbox': [1003.0, 549.0, 1447.0, 950.0], 'conversations': [{'from': 'human', 'value': 'Visualize the tracking trajectory of panda rolling around in the video.'}, {'from': 'gpt', 'value': 'Answer: <track_begin><tracking> .'}]}


0it [00:00, ?it/s]

/tmp/ipykernel_20349/3907469626.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(tt['sam_gt_embs'])
14259it [00:25, 551.95it/s]


12933
1326


In [2]:
import os
import torch
wrong = []
for dd in os.listdir('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_feature/'):
    # print(dd)
    try:
        feature = os.path.join('/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_feature/', dd)
        feature = torch.load(feature)
        # print(feature.shape)
        if feature.shape != torch.Size([1, 3, 256]):
            print(dd)
            raise UnboundLocalError
            
    except:
        # print(dd)
        wrong.append(dd)
print(len(wrong))


/tmp/ipykernel_63198/837561494.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(feature)


688
